<a href="https://colab.research.google.com/github/Apoorva-Menon/ExtremeYears/blob/milestone4/preprocessing/NASA_EDA_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
import pandas as pd

In [4]:
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/nasa-viz/PSCompPars_2026.10.27_11.16.54.csv', skiprows=328)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
df.head()

,rowid,pl_name,hostname,pl_letter,hd_name,hip_name,tic_id,gaia_dr2_id,gaia_dr3_id,sy_snum,...,pl_nnotes,st_nphot,st_nrvc,st_nspec,pl_nespec,pl_ntranspec,pl_ndispec,pl_nobs_jwst_tran,pl_nobs_jwst_e,pl_nobs_jwst_pc
0,1,11 Com b,11 Com,b,HD 107383,HIP 60202,TIC 72437047,Gaia DR2 3946945413106333696,Gaia DR3 3946945413106333696,2,...,2,1,2,0,0,0,0,0,0,0
1,2,11 UMi b,11 UMi,b,HD 136726,HIP 74793,TIC 230061010,Gaia DR2 1696798367260229376,Gaia DR3 1696798367260229376,1,...,0,1,1,0,0,0,0,0,0,0
2,3,14 And b,14 And,b,HD 221345,HIP 116076,TIC 333225860,Gaia DR2 1920113512486282240,Gaia DR3 1920113512486282240,1,...,0,1,1,0,0,0,0,0,0,0
3,4,14 Her b,14 Her,b,HD 145675,HIP 79248,TIC 219483057,Gaia DR2 1385293808145621504,Gaia DR3 1385293808145621504,1,...,1,1,4,1,0,0,0,0,0,0
4,5,16 Cyg B b,16 Cyg B,b,HD 186427,HIP 96901,TIC 27533327,Gaia DR2 2135550755683407232,Gaia DR3 2135550755683407232,3,...,5,1,4,3,0,0,0,0,0,0


In [6]:
pd.set_option("display.max_rows", None)

print(df.isna().sum())

rowid                      0
pl_name                    0
hostname                   0
pl_letter                  0
hd_name                 5383
hip_name                5284
tic_id                   307
gaia_dr2_id              363
gaia_dr3_id              398
sy_snum                    0
sy_pnum                    0
sy_mnum                    0
cb_flag                    0
discoverymethod            0
disc_year                  0
disc_refname               0
disc_pubdate               0
disc_locale                0
disc_facility              0
disc_telescope             0
disc_instrument            0
rv_flag                    0
pul_flag                   0
ptv_flag                   0
tran_flag                  0
ast_flag                   0
obm_flag                   0
micro_flag                 0
etv_flag                   0
ima_flag                   0
dkin_flag                  0
pl_controv_flag            0
pl_orbper                354
pl_orbpererr1            850
pl_orbpererr2 

In [7]:
df_filtered = df[df["pl_controv_flag"] == 0].copy()

print(df_filtered.isna().sum())

rowid                      0
pl_name                    0
hostname                   0
pl_letter                  0
hd_name                 5352
hip_name                5254
tic_id                   307
gaia_dr2_id              362
gaia_dr3_id              396
sy_snum                    0
sy_pnum                    0
sy_mnum                    0
cb_flag                    0
discoverymethod            0
disc_year                  0
disc_refname               0
disc_pubdate               0
disc_locale                0
disc_facility              0
disc_telescope             0
disc_instrument            0
rv_flag                    0
pul_flag                   0
ptv_flag                   0
tran_flag                  0
ast_flag                   0
obm_flag                   0
micro_flag                 0
etv_flag                   0
ima_flag                   0
dkin_flag                  0
pl_controv_flag            0
pl_orbper                350
pl_orbpererr1            843
pl_orbpererr2 

In [8]:
print("Total planets:", len(df))
print("Controversial planets:", len(df) - len(df_filtered))
print("Non-controversial planets:", len(df_filtered))

Total planets: 6451
Controversial planets: 58
Non-controversial planets: 6393


In [9]:
selected_cols = [
    "pl_name",
    "hostname",
    "pl_rade",
    "pl_bmasse",
    "discoverymethod",
    "pl_eqt",
    "st_lum",
    "pl_orbeccen",
    "st_mass",
    "pl_orbper",
    "pl_orbsmax",
    "st_teff",
    "pl_insol"
]

missing = pd.DataFrame({
    "Available (Original)": df[selected_cols].notna().sum(),
    "Missing (Original)": df[selected_cols].isna().sum(),
    "Missing % (Original)": df[selected_cols].isna().mean() * 100,
    "Available (CF applied)": df_filtered[selected_cols].notna().sum(),
    "Missing (CF applied)": df_filtered[selected_cols].isna().sum(),
    "Missing % (CF applied)": df_filtered[selected_cols].isna().mean() * 100
})

display(missing.round(2))

,Available (Original),Missing (Original),Missing % (Original),Available (CF applied),Missing (CF applied),Missing % (CF applied)
pl_name,6451,0,0.00,6393,0,0.00
hostname,6451,0,0.00,6393,0,0.00
pl_rade,6401,50,0.78,6349,44,0.69
pl_bmasse,6421,30,0.47,6368,25,0.39
discoverymethod,6451,0,0.00,6393,0,0.00
pl_eqt,5911,540,8.37,5865,528,8.26
st_lum,6125,326,5.05,6071,322,5.04
pl_orbeccen,5383,1068,16.56,5334,1059,16.56
st_mass,6442,9,0.14,6386,7,0.11
pl_orbper,6097,354,5.49,6043,350,5.47


## Shared preprocessing for Observable

Retain the existing non-controversial NASA planet rows and source columns. Standardize the numeric fields needed for the scientific metric and calculate irradiation_ratio (periastron irradiation / apastron irradiation). Missing or invalid eccentricity produces missing R, not an imputed value. No chart-specific filtering, binning, or ranking is performed in this pipeline.

In [10]:
import numpy as np
from pathlib import Path

# Start from the original, non-controversial planet population.
planets = df_filtered.copy()

# Standardize the numeric inputs needed for the scientific calculation.
for column in ["pl_orbper", "pl_orbeccen"]:
    planets[column] = pd.to_numeric(planets[column], errors="coerce")
    planets.loc[~np.isfinite(planets[column]), column] = np.nan

# R is defined for bound elliptical orbits with 0 <= e < 1.
e = planets["pl_orbeccen"]
valid_e = e.notna() & e.ge(0) & e.lt(1)
planets["irradiation_ratio"] = np.nan
planets.loc[valid_e, "irradiation_ratio"] = (
    (1 + e.loc[valid_e]) / (1 - e.loc[valid_e])
) ** 2

# A source-recorded e=0 is retained as 0, yielding R=1.
# No orbital-period filter is applied here: charts may have different requirements.
print("Processed rows:", len(planets))
print("Duplicate planet names:", int(planets["pl_name"].duplicated().sum()))
display(planets[["pl_name", "pl_orbper", "pl_orbeccen", "irradiation_ratio"]].head())

Processed rows: 6393
Duplicate planet names: 0


,pl_name,pl_orbper,pl_orbeccen,irradiation_ratio
0,11 Com b,323.21000,0.2380,2.639559
1,11 UMi b,516.21997,0.0800,1.378072
2,14 And b,186.76000,0.0000,1.000000
3,14 Her b,1766.41000,0.3683,4.691815
4,16 Cyg B b,798.50000,0.6800,27.562500


### Preprocessing audit (not chart eligibility)

These checks describe source completeness and mathematical validity; they do not add flags to the exported CSV. Observable will determine eligible rows independently for each visualization.

In [11]:
period = planets["pl_orbper"]
ratio = planets["irradiation_ratio"]
audit = pd.DataFrame([
    {"check": "Non-controversial planet rows", "count": len(planets)},
    {"check": "Finite, positive orbital periods", "count": int((period.notna() & period.gt(0)).sum())},
    {"check": "Valid eccentricities (0 <= e < 1)", "count": int(valid_e.sum())},
    {"check": "Finite irradiation ratios (R >= 1)", "count": int((ratio.notna() & np.isfinite(ratio) & ratio.ge(1)).sum())},
    {"check": "Recorded eccentricity e = 0", "count": int(e.eq(0).sum())},
    {"check": "Duplicate planet names", "count": int(planets["pl_name"].duplicated().sum())},
])
display(audit)

,check,count
0,Non-controversial planet rows,6393
1,"Finite, positive orbital periods",6043
2,Valid eccentricities (0 <= e < 1),5334
3,Finite irradiation ratios (R >= 1),5334
4,Recorded eccentricity e = 0,3146
5,Duplicate planet names,0


### Scientific spot-check

Inspect the largest calculated ratios to flag unusually high eccentricities for later scientific validation. This preview does not set the chart ranking or exclude any planet.

In [12]:
display(
    planets.loc[planets["irradiation_ratio"].notna(),
                ["pl_name", "pl_orbper", "pl_orbeccen", "irradiation_ratio"]]
    .sort_values(["irradiation_ratio", "pl_name"], ascending=[False, True])
    .head(10)
)

,pl_name,pl_orbper,pl_orbeccen,irradiation_ratio
939,HD 20782 b,597.064300,0.95000,1521.000000
1069,HD 28254 b,1333.000000,0.95000,1521.000000
1413,HIP 66074 b,310.900000,0.94800,1403.366864
5235,TIC 241249530 b,165.771900,0.94120,1089.898144
1275,HD 80606 b,111.436765,0.93183,803.065972
259,GJ 3222 b,10.666690,0.92900,738.155326
1277,HD 80869 b,1752.920670,0.92400,640.889197
1053,HD 26161 b,10942.290130,0.92200,607.180145
3142,Kepler-1704 b,988.881120,0.92000,576.000000
1143,HD 4113 b,526.523900,0.89900,353.514459


### Export for Observable

Export the non-controversial NASA rows with their original source columns and the calculated irradiation_ratio. The CSV contains no chart-specific eligibility flags. In Observable, filter per chart, construct logarithmic histogram bins, rank planets, and implement tooltips and interactions.

In [13]:
output_dir = Path("/content/extreme_years_exports") if Path("/content").exists() else Path("extreme_years_exports")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "planets_processed.csv"
planets.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Rows:", len(planets), "Columns:", len(planets.columns))

# Optional Colab download:
# from google.colab import files
# files.download(str(output_path))

Saved: /content/extreme_years_exports/planets_processed.csv
Rows: 6393 Columns: 326


### Preprocessing required for Q3/Q4

In [14]:
# Fill in missing orbit size and starlight using physics.
# pl_orbsmax_source column records where each value came from: archive or kepler_third_law

# Orbit size (AU) from Kepler's third law: a = cube root of (star mass × period in years²)
planets["pl_orbsmax_source"] = np.where(planets["pl_orbsmax"].notna(), "archive", None)
need_a = planets["pl_orbsmax"].isna() & planets["pl_orbper"].gt(0) & planets["st_mass"].gt(0)
planets.loc[need_a, "pl_orbsmax"] = np.cbrt(
    planets.loc[need_a, "st_mass"] * (planets.loc[need_a, "pl_orbper"] / 365.25) ** 2
)
planets.loc[need_a, "pl_orbsmax_source"] = "kepler_third_law"

In [15]:

# pl_insol_source column records where each value came from: archive or from_st_lum
# Starlight at the average distance (Earth = 1) from the inverse-square law: S = L / a^2
# st_lum is stored as log10(L / L_sun), so 10 ** st_lum gives L in solar units.
planets["pl_insol_source"] = np.where(planets["pl_insol"].notna(), "archive", None)
need_s = planets["pl_insol"].isna() & planets["st_lum"].notna() & planets["pl_orbsmax"].gt(0)
planets.loc[need_s, "pl_insol"] = 10 ** planets.loc[need_s, "st_lum"] / planets.loc[need_s, "pl_orbsmax"] ** 2
planets.loc[need_s, "pl_insol_source"] = "from_st_lum"

In [16]:
BURST_SHARE = 0.25          # below 25% of the year near peak is a burst year
EXCLUDE_UPPER_LIMITS = True # True = eccentricities reported only as limits are not usable

def above_half_peak_time(ecc):
    """Share of the year a planet spends at or above half of its peak starlight.
    Uses only eccentricity. e = 0 (circular) gives 1, i.e. the whole year."""
    ecc = np.asarray(ecc, dtype=float)
    share = np.ones_like(ecc)                                 # e = 0 -> whole year
    stretched = ecc > 0
    e_s = ecc[stretched]
    r_half = np.sqrt(2) * (1 - e_s)                             # distance where starlight = 1/2 peak (units of a)
    E_half = np.arccos(np.clip((1 - r_half) / e_s, -1, 1))      # orbit position at that distance
    share[stretched] = (E_half - e_s * np.sin(E_half)) / np.pi   # Kepler's equation
    return share

In [17]:
e = planets["pl_orbeccen"]
valid_e = e.notna() & e.ge(0) & e.lt(1)
if EXCLUDE_UPPER_LIMITS:
    valid_e &= planets["pl_orbeccenlim"].fillna(0).eq(0)
insol = planets["pl_insol"]

In [18]:
# Eccentricity with its filters
planets["e"] = e.where(valid_e)

# Quality label - defines whether the eccentricity is 0 (which means it was not measured) or if it is measured
planets["e_quality"] = np.select(
    [valid_e & e.gt(0), valid_e & e.eq(0)],
    ["measured", "recorded_zero"],
    default="not_usable")

In [19]:
# Peak starlight at closest approach: S_peak = pl_insol / (1 − e)^2
has_peak = valid_e & insol.gt(0)
planets["S_peak"] = np.nan
planets.loc[has_peak, "S_peak"] = insol[has_peak] / (1 - e[has_peak]) ** 2

In [20]:
# Share of the year near peak, days near peak, and burst / prolonged label
planets["above_half_peak_time"] = np.nan
planets.loc[valid_e, "above_half_peak_time"] = above_half_peak_time(e[valid_e])
planets["days_bright"] = planets["above_half_peak_time"] * planets["pl_orbper"]
planets["exposure"] = None
planets.loc[valid_e, "exposure"] = np.where(
    planets.loc[valid_e, "above_half_peak_time"] < BURST_SHARE, "burst", "prolonged"
)

In [21]:
display(planets.loc[planets["e_quality"] != "not_usable",
        ["pl_name", "pl_orbeccen", "e_quality", "pl_insol", "S_peak",
         "above_half_peak_time", "days_bright", "exposure"]].head())

,pl_name,pl_orbeccen,e_quality,pl_insol,S_peak,above_half_peak_time,days_bright,exposure
0,11 Com b,0.2380,measured,68.5387,118.039108,0.534147,172.641655,prolonged
1,11 UMi b,0.0800,measured,114.8502,135.692580,1.000000,516.219970,prolonged
2,14 And b,0.0000,recorded_zero,115.1634,115.163400,1.000000,186.760000,prolonged
3,14 Her b,0.3683,measured,0.0873,0.218772,0.294283,519.823978,prolonged
4,16 Cyg B b,0.6800,measured,0.4540,4.433594,0.073732,58.875032,burst


In [22]:
STORY_COLS = ["pl_name", "hostname", "discoverymethod", "pl_orbper", "pl_insol",
              "e", "e_quality", "S_peak", "irradiation_ratio",
              "above_half_peak_time", "days_bright", "exposure"]

story = planets.loc[valid_e, STORY_COLS].round(6)
story_path = output_dir / "planets_story.csv"
story.to_csv(story_path, index=False)

print("Saved:", story_path)
print(f"Rows: {len(story):,}   measured: {(story.e_quality == 'measured').sum():,}   recorded zero: {(story.e_quality == 'recorded_zero').sum():,}")

from google.colab import files
files.download(str(story_path))

Saved: /content/extreme_years_exports/planets_story.csv
Rows: 5,037   measured: 1,892   recorded zero: 3,145


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# Run after the Q3/Q4 cells have filled orbit size/flux and calculated bright days.
# Keep their shared planets table unchanged.
q56_e = planets["pl_orbeccen"]
q56_eligible = (
    planets["pl_orbper"].gt(0)
    & planets["pl_orbsmax"].gt(0)
    & planets["pl_insol"].gt(0)
    & q56_e.ge(0) & q56_e.lt(1)
    & planets["pl_orbeccenlim"].eq(0)
)
q56_base = planets.loc[q56_eligible].copy()
print(f"Q5/Q6 eligible planets: {len(q56_base):,}")

Q5/Q6 eligible planets: 5,018


## Q5/Q6 — Characteristics and extreme planetary years

**Q5:** How do planetary, orbital, and stellar characteristics relate to irradiation extremes?

**Q6:** Which five exoplanets have the wildest years, and what makes each extraordinary?

We use irradiation contrast R = ((1 + e) / (1 − e))². This measures variation in incoming starlight, not temperature or habitability. These cells prepare the data for the four Q5/Q6 D3 charts in Observable.

In [24]:
q56_shared_export = planets.copy(deep=True)

### 1. Select the Q5/Q6 population
Reuse the shared preprocessing’s filled orbit sizes and reference fluxes. Retain positive period, orbit size, and flux, with 0 ≤ eccentricity < 1 and an eccentricity limit flag of zero

In [25]:
# Run after the Q3/Q4 cells have filled orbit size/flux and calculated bright days.
# Keep their shared planets table unchanged.
q56_e = planets["pl_orbeccen"]
q56_eligible = (
    planets["pl_orbper"].gt(0)
    & planets["pl_orbsmax"].gt(0)
    & planets["pl_insol"].gt(0)
    & q56_e.ge(0) & q56_e.lt(1)
    & planets["pl_orbeccenlim"].eq(0)
)
q56_base = planets.loc[q56_eligible].copy()
print(f"Q5/Q6 eligible planets: {len(q56_base):,}")

Q5/Q6 eligible planets: 5,018


### 2. Prepare the chart fields
Reuse the existing R, peak flux, and bright days. Add luminosity in solar units, minimum flux, and R at valid published eccentricity-error endpoints. Missing or invalid error bounds remain blank; these are not rank confidence intervals.

In [26]:
q56_e = q56_base["pl_orbeccen"]

# The existing cells already supply R, S_peak, and days_bright.
q56_base["lum_solar"] = 10 ** q56_base["st_lum"]
q56_base["S_min"] = q56_base["pl_insol"] / (1 + q56_e) ** 2

# Transform valid published error endpoints; leave unavailable bounds blank.
q56_lower_e = q56_e + q56_base["pl_orbeccenerr2"]
q56_upper_e = q56_e + q56_base["pl_orbeccenerr1"]
q56_valid_lower = q56_lower_e.ge(0) & q56_lower_e.le(q56_e)
q56_valid_upper = q56_upper_e.ge(q56_e) & q56_upper_e.lt(1)

q56_base["R_lower"] = np.nan
q56_base["R_upper"] = np.nan
q56_base.loc[q56_valid_lower, "R_lower"] = (
    (1 + q56_lower_e.loc[q56_valid_lower]) / (1 - q56_lower_e.loc[q56_valid_lower])
) ** 2
q56_base.loc[q56_valid_upper, "R_upper"] = (
    (1 + q56_upper_e.loc[q56_valid_upper]) / (1 - q56_upper_e.loc[q56_valid_upper])
) ** 2

### 3. Verify the results
Check unique planets, the flux-ratio identity, and the circular-orbit case. Show the actual five highest central R values and their inputs. With this source export, expect 5,018 eligible planets: 1,876 positive eccentricities and 3,142 recorded zeros.

In [27]:
q56_positive_count = int((q56_base["pl_orbeccen"] > 0).sum())
q56_zero_count = int((q56_base["pl_orbeccen"] == 0).sum())
print(f"Q3 comparison: {q56_positive_count:,} positive eccentricities; {q56_zero_count:,} recorded zeros")

assert q56_base["pl_name"].is_unique
assert q56_positive_count + q56_zero_count == len(q56_base)
assert np.allclose(
    q56_base["S_peak"] / q56_base["S_min"],
    q56_base["irradiation_ratio"]
)
assert q56_base["above_half_peak_time"].between(0, 1).all()
q56_circular = q56_base["pl_orbeccen"].eq(0)
assert q56_base.loc[q56_circular, "irradiation_ratio"].eq(1).all()
assert q56_base.loc[q56_circular, "above_half_peak_time"].eq(1).all()

q56_ranked = q56_base.sort_values(
    ["irradiation_ratio", "pl_name"], ascending=[False, True]
).head(5)
q56_sample_columns = [
    "pl_name", "pl_orbeccen", "pl_orbeccenerr2", "pl_orbeccenerr1",
    "pl_orbper", "pl_orbsmax", "irradiation_ratio", "R_lower", "R_upper", "days_bright"
]
display(q56_ranked[q56_sample_columns])

Q3 comparison: 1,876 positive eccentricities; 3,142 recorded zeros


,pl_name,pl_orbeccen,pl_orbeccenerr2,pl_orbeccenerr1,pl_orbper,pl_orbsmax,irradiation_ratio,R_lower,R_upper,days_bright
939,HD 20782 b,0.95000,-0.00100,0.00100,597.064300,1.3649,1521.000000,1460.438677,1585.339858,2.262717
1069,HD 28254 b,0.95000,-0.04000,0.03000,1333.000000,2.4500,1521.000000,450.382716,9801.000000,5.051719
1413,HIP 66074 b,0.94800,-0.00400,0.00400,310.900000,0.7990,1403.366864,1205.081633,1653.777778,1.251049
5235,TIC 241249530 b,0.94120,-0.00090,0.00090,165.771900,0.6410,1089.898144,1056.304440,1125.086851,0.805221
1275,HD 80606 b,0.93183,-0.00014,0.00014,111.436765,0.4603,803.065972,799.661694,806.491538,0.679371


In [28]:
q56_shared_export = planets.copy(deep=True)
q56_lookup = q56_base.set_index("pl_name")

q56_shared_export["q56_eligible"] = (
    q56_shared_export["pl_name"].isin(q56_lookup.index).astype(int)
)

for q56_column in ["lum_solar", "R_lower", "R_upper"]:
    q56_shared_export[q56_column] = (
        q56_shared_export["pl_name"].map(q56_lookup[q56_column])
    )

q56_shared_export.to_csv(output_path, index=False)

from google.colab import files as q56_files
q56_files.download(str(output_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>